# AI-асистент аналітика: демонстрація на реальних даних
Дані: реєстрації та платежі мобільного застосунку (квітень-травень 2024). Pandas рахує всі числа, LLM їх інтерпретує.

In [7]:
import os, json
os.environ.setdefault('LLM_API_KEY', '')  # встав ключ тут або задай змінною середовища
from ai_analyst.data import load_demo
from ai_analyst.llm import LLM
from ai_analyst.eda import auto_eda
from ai_analyst.nl2pandas import ask
from ai_analyst.anomaly import explain_metric_change
from ai_analyst.summary import kpi_snapshot, executive_summary

llm = LLM()
tables = load_demo()
{k: v.shape for k, v in tables.items()}

{'registrations': (202383, 7), 'payments': (11330, 6)}

## 1. Автоматичний EDA

In [8]:
r = auto_eda(tables['payments'], llm, 'payments')
print(json.dumps(r['interpretation'], ensure_ascii=False, indent=2))
print('Числа поза профілем:', r['unverified_numbers'])

{
  "quality_summary": "Дані мають повну заповненість і відсутність дублікатів, проте значна частка аутлайерів у колонках user_id (19.98%) та amount_usd (22.6%) може впливати на аналіз.",
  "issues": [
    {
      "column": "user_id",
      "problem": "високий відсоток аутлайерів",
      "severity": "high",
      "evidence": "19.98"
    },
    {
      "column": "amount_usd",
      "problem": "високий відсоток аутлайерів",
      "severity": "high",
      "evidence": "22.6"
    }
  ],
  "key_findings": [
    "Всі 6 колонок заповнені без пропусків і без негативних чи нульових значень",
    "Кількість унікальних user_id – 7781, що свідчить про різноманітність користувачів",
    "feature_type_id має лише два можливих значення (1 і 2)",
    "country_name містить 109 унікальних країн, провідні – Ukraine (3245) та United States of America (1826)",
    "app: 89.8% записів – mobile"
  ],
  "next_steps": [
    "Проаналізувати причини аутлайерів у user_id та amount_usd (можливі помилки вводу або о

## 2. Питання природною мовою -> pandas (код видно й можна перевірити)

In [9]:
r = ask('Покажи топ-10 країн за виручкою', tables, llm)
print(r['code']); print(r['explanation']); r['result']

result = (payments.groupby('country_name')['amount_usd']
                .sum()
                .reset_index(name='revenue')
                .sort_values('revenue', ascending=False)
                .head(10))
Групуємо payments за country_name, підсумовуємо amount_usd, сортуємо за сумою у спадаючому порядку і беремо перші 10 рядків.


,country_name,revenue
100,Ukraine,22954.52
103,United States of America,19141.17
90,South Africa,6053.40
17,Chile,5513.70
91,Spain,4895.21
102,United Kingdom of Great Britain and Northern I...,3966.89
53,Kazakhstan,3669.68
3,Argentina,2957.92
36,Germany,2689.54
78,Poland,2210.52


In [10]:
r = ask('Яка середня температура у Києві?', tables, llm)  # нелогічний запит: модель має відмовитись
print(r['refused'], r['explanation'])

True Неможливо, оскільки в наданих таблицях немає колонок з даними про температуру


## 3. Пояснення зміни виручки між двома тижнями

In [11]:
base, target = ('2024-04-15', '2024-04-21'), ('2024-04-22', '2024-04-28')
r = explain_metric_change(tables['payments'], 'pay_dt', 'amount_usd', ['country_name', 'app', 'feature_type_id'], base, target, llm)
print(json.dumps(r['facts'], ensure_ascii=False, indent=1)[:1500])
print(json.dumps(r['explanation'], ensure_ascii=False, indent=2))
print('Числа поза фактами:', r['unverified_numbers'])

{
 "metric": "sum(amount_usd)",
 "base_period": [
  "2024-04-15",
  "2024-04-21"
 ],
 "target_period": [
  "2024-04-22",
  "2024-04-28"
 ],
 "base_value": 13972.71,
 "target_value": 25022.06,
 "delta": 11049.35,
 "delta_pct": 79.08,
 "days_base": 7,
 "days_target": 7,
 "segments": {
  "country_name": [
   {
    "value": "Ukraine",
    "base": 3926.47,
    "target": 6190.8,
    "delta": 2264.33,
    "share_of_delta_pct": 20.5
   },
   {
    "value": "South Africa",
    "base": 726.21,
    "target": 1900.54,
    "delta": 1174.33,
    "share_of_delta_pct": 10.6
   },
   {
    "value": "Kazakhstan",
    "base": 390.66,
    "target": 1297.83,
    "delta": 907.17,
    "share_of_delta_pct": 8.2
   },
   {
    "value": "Spain",
    "base": 453.93,
    "target": 1345.89,
    "delta": 891.96,
    "share_of_delta_pct": 8.1
   },
   {
    "value": "Chile",
    "base": 903.4,
    "target": 1713.0,
    "delta": 809.6,
    "share_of_delta_pct": 7.3
   }
  ],
  "app": [
   {
    "value": "mobile",
   

## 4. Executive summary

In [12]:
m = kpi_snapshot(tables['registrations'], tables['payments'], base, target)
r = executive_summary(m, llm)
print(json.dumps(r['summary'], ensure_ascii=False, indent=2))
print('Числа поза метриками:', r['unverified_numbers'])

{
  "what_happened": "За період 2024‑04‑22 – 2024‑04‑28 кількість нових користувачів зросла до 31 879, що на 58.5 % більше ніж у попередньому тижні, дохід піднявся до 25 022.06 USD (+79.1 %), а кількість платників збільшилася до 2 259 (+100.1 %). Середній чек знизився до 8.1 USD (‑6.5 %).",
  "why_it_matters": "Зростання користувачів і платників свідчить про успішне залучення аудиторії, а падіння середнього чеку може скоротити довгострокову прибутковість.",
  "recommendations": [
    "Проаналізувати, які канали залучення принесли найбільший приріст нових користувачів, і посилити інвестиції в них",
    "Вивчити причини зниження середнього чеку та розробити стратегії підвищення середньої вартості транзакції",
    "Оптимізувати монетизаційні пропозиції для нових платників, щоб підтримати темпи зростання доходу"
  ],
  "risks": [
    "Дані охоплюють лише по одному тижневому інтервалу, що може не відображати довгострокові тенденції",
    "Відсутня деталізація за сегментами користувачів, том